# Attention, By Hand (and by NumPy)

Companion notebook for the **Attention** module of the LLM Fundamentals course (`llm:attention`).

This walks through the same 3-token "cat chased mouse" example from the course, but lets you verify every step by running real code instead of a calculator. Run the cells top to bottom.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## Step 0 — the toy example

Sentence: "cat chased mouse" — we're computing attention output for the token "chased".

Real models use 64-128+ dimensions per attention head; this toy example uses 2 dimensions so every number stays visible.

In [ ]:
query_chased = np.array([1.0, 0.0])

keys = {
    "cat":    np.array([1.0, 0.0]),
    "chased": np.array([0.0, 1.0]),
    "mouse":  np.array([0.8, 0.2]),
}

values = {
    "cat":    np.array([2.0, 5.0]),
    "chased": np.array([0.0, 1.0]),
    "mouse":  np.array([4.0, 1.0]),
}

## Step 1 — raw scores (Query · Key)

In [ ]:
scores = {tok: float(query_chased @ k) for tok, k in keys.items()}
scores

Expected: `cat=1.0, chased=0.0, mouse=0.8` — matches the course walkthrough.

## Step 2 — softmax turns scores into weights that sum to 1

In [ ]:
def softmax(x):
    exps = np.exp(x)
    return exps / exps.sum()

tokens = list(scores.keys())
score_vec = np.array([scores[t] for t in tokens])
weights = softmax(score_vec)
dict(zip(tokens, weights))

Expected: roughly `cat=0.46, chased=0.17, mouse=0.37` — same numbers as the course's hand-computed example.

## Step 3 — output = weighted sum of Values

In [ ]:
value_matrix = np.array([values[t] for t in tokens])
output = weights @ value_matrix
output

Expected: approximately `[2.40, 2.84]`.

"chased" ends up attending most to "cat" (weight ≈ 0.46) — plausible, since in "cat chased mouse" the subject is highly relevant to the verb.

## Exercise — try a different query

The course's follow-up exercise: recompute using `q = [0.0, 1.0]` instead (pretend "chased" is now "looking for its subject's target"). Which token gets the highest weight, and does that match your intuition?

Edit `query_chased` below and re-run steps 1-3, or just use this cell.

In [ ]:
def run_attention(query, keys, values):
    tokens = list(keys.keys())
    score_vec = np.array([float(query @ keys[t]) for t in tokens])
    weights = softmax(score_vec)
    value_matrix = np.array([values[t] for t in tokens])
    output = weights @ value_matrix
    return dict(zip(tokens, weights)), output

new_query = np.array([0.0, 1.0])
new_weights, new_output = run_attention(new_query, keys, values)
print("weights:", new_weights)
print("output: ", new_output)

## Going further

- Try adding a fourth token to the sentence and see how the weights redistribute.
- Increase the vector dimensionality to 4 or 8 and confirm the same three steps still work unchanged — attention doesn't care about dimensionality.
- Compare this by-hand implementation to `torch.nn.functional.scaled_dot_product_attention` (note the real formula also divides scores by `sqrt(d_k)` before the softmax, which this toy example skips for simplicity).